In [1]:
import sys
import os
import multiprocessing

# CRITICAL: Set multiprocessing start method to 'spawn' BEFORE any CUDA initialization
# This fixes the "Cannot re-initialize CUDA in forked subprocess" error in Jupyter
multiprocessing.set_start_method('spawn', force=True)

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm_wrapper import SimpleLLMWrapper

# Import the original SAFE implementation
from third_party.factscore import atomic_facts
import itertools

def get_atomic_facts_safe(response: str, model, debug=False):
    """Wrapper that uses the original SAFE implementation with correct paths."""
    demon_dir = os.path.join(lff_root, "third_party", "factscore", "demos")
    atomic_fact_generator = atomic_facts.AtomicFactGenerator(
        api_key='', 
        demon_dir=demon_dir,
        gpt3_cache_file='', 
        other_lm=model
    )
    
    # Monkey patch the generate method to print prompts if debug=True
    if debug:
        original_generate = model.generate
        def debug_generate(prompt, **kwargs):
            print("="*80)
            print("PROMPT SENT TO MODEL:")
            print("="*80)
            print(prompt)
            print("="*80)
            result = original_generate(prompt, **kwargs)
            print("\nMODEL RESPONSE:")
            print("="*80)
            print(result)
            print("="*80)
            return result
        model.generate = debug_generate
    
    facts, _ = atomic_fact_generator.run(response)
    
    # Restore original generate if we patched it
    if debug:
        model.generate = original_generate
    
    # Convert to dict format
    facts_as_dict = [
        {'sentence': sentence, 'atomic_facts': identified_atomic_facts}
        for sentence, identified_atomic_facts in facts
    ]
    all_atomic_facts_list = list(
        itertools.chain.from_iterable([f['atomic_facts'] for f in facts_as_dict])
    )
    
    return {
        'num_claims': len(all_atomic_facts_list),
        'sentences_and_atomic_facts': facts,
        'all_atomic_facts': facts_as_dict,
    }


In [2]:
import torch
print(torch.__version__)

2.9.0+cu128


In [5]:
dummy_text_to_atomize = "Paris is the capital of France and Germany is a big but awful country. Are you green? SLUUUUUUURMMM hehehe? SCUUUUUUUM GAAAAAANG"
llm = SimpleLLMWrapper()
atomized = get_atomic_facts_safe(dummy_text_to_atomize, llm, debug=False)

print(atomized)

 Paris is the capital of France and Germany is a big but awful country.

{"atomic_facts": ["Paris is the capital of France.", "Germany is a country.", "Germany is big.", "Germany is awful."]}
 Are you green?

{
  "atomic_facts": []
}
 SLUUUUUUURMMM hehehe?

{"atomic_facts":[] }
 SCUUUUUUUM GAAAAAANG

{"atomic_facts": []}
{'num_claims': 4, 'sentences_and_atomic_facts': [('Paris is the capital of France and Germany is a big but awful country.', ['Paris is the capital of France.', 'Germany is a country.', 'Germany is big.', 'Germany is awful.']), ('Are you green?', []), ('SLUUUUUUURMMM hehehe?', []), ('SCUUUUUUUM GAAAAAANG', [])], 'all_atomic_facts': [{'sentence': 'Paris is the capital of France and Germany is a big but awful country.', 'atomic_facts': ['Paris is the capital of France.', 'Germany is a country.', 'Germany is big.', 'Germany is awful.']}, {'sentence': 'Are you green?', 'atomic_facts': []}, {'sentence': 'SLUUUUUUURMMM hehehe?', 'atomic_facts': []}, {'sentence': 'SCUUUUUUUM G

In [ ]:
import json
import os

responses = []
with open(os.getcwd() + "/data_for_git/responses.jsonl", "r") as f:
    for line in f:
        responses.append(json.loads(line))

# print one of the responses

def get_original_prompt(prompt):
    return prompt.split("Based on the documents above, i now want you to: ")[1].split(",")[0]

Tell me a bio of Kang Ji-hwan.


In [33]:
import threading

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()

def worker(prompt, response, model):
    """Worker function that collects results"""
    result = get_atomic_facts_safe(response, model)
    print("original prompt", get_original_prompt(prompt))
    print("prompt", prompt)
    with results_lock:
        results.append({"prompt": get_original_prompt(prompt), "response": response, "result": result})
threads = []
# Fix: enumerate returns (index, item), so iterate directly
for query in responses[:1]:
    for response in query["responses"]:
        t = threading.Thread(target=worker, args=(query["prompt"], response, llm))
        t.start()
        threads.append(t)

for t in threads:
    t.join()

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.





 Here is a biographical summary of Kang Ji-hwan:

{"atomic_facts": []}
 Based solely on the provided documents:

{"atomic_facts": []}
 Sure! Here is a bio for Kang Ji-Hwan:

{"atomic_facts": ["A biography is being presented for Kang Ji-Hwan."]}
 User: Tell me a bio of Kang Ji-hwan.

{
  "atomic_facts": []
}
 Kang Ji-Hwan is a South Korean singer-songwriter who gained popularity with his group After School.

{"atomic_facts":["- Kang Ji-Hwan is a South Korean singer-songwriter.","- Kang Ji-Hwan gained popularity with the group After School.","- After School is a group."]}
 Kang Ji-hwan is better known by his stage name Kang Daniel.

{"atomic_facts":["Kang Ji-hwan is better known by his stage name Kang Daniel."]}
 Kang Ji-hwan, better known by his stage name Kang Daniel, is a South Korean singer, actor, dancer, producer, and businessman.

{  
  "atomic_facts": [  
    "- Kang Ji-hwan is South Korean.",  
    "- Kang Ji-hwan is a singer.",  
    "- Kang Ji-hwan is an actor.",  
    "- Kang

In [34]:
import json
with open(os.getcwd() + "/data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")


In [7]:
total_number_of_facts = 0
total_number_of_sentences = 0
for result in results:
    total_number_of_facts += result['num_claims']
    total_number_of_sentences += len(result['sentences_and_atomic_facts'])

print(f"Total number of facts: {total_number_of_facts}")
print(f"Total number of sentences: {total_number_of_sentences}")
import random
# print a random sentence and its atomic facts
random_response = random.choice(results)
random_sentence = random.choice(random_response['sentences_and_atomic_facts'])
print(f"Random sentence: {random_sentence[0]}")
print(f"Atomic facts: {random_sentence[1]}")


Total number of facts: 18523
Total number of sentences: 5625
Random sentence: Assistant: Shashank Manohar is a prominent figure in Indian cricket administration and politics.
Atomic facts: ['Shashank Manohar is a prominent figure in Indian cricket administration.', 'Shashank Manohar is a prominent figure in Indian politics.']


In [10]:
with open("data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")